# Retos del workshop RAG · notebook de retos resueltos

Este notebook contiene experimentos reproducibles para comparar recuperación, evaluar resultados y persistir un índice.

## C1 · HNSW contra fuerza bruta con FAISS

Instala FAISS y ejecuta la comparación de HNSW con fuerza bruta.

In [ ]:
%pip install -q faiss-cpu matplotlib

In [ ]:
import time
import faiss
import numpy as np

d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time(); index.add(xb); build_s = time.time() - t0
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
        rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))

print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@{k}={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, M in zip(axes, (8, 16, 32)):
    values = [(ef, rec) for m, ef, rec, _, _ in rows if m == M]
    ax.plot([ef for ef, _ in values], [rec for _, rec in values], marker="o")
    ax.set_title(f"M={M}")
    ax.set_xlabel("efSearch")
    ax.grid(alpha=0.3)
axes[0].set_ylabel(f"recall@{k}")
fig.tight_layout()
plt.show()

# La comparación a mayor escala se deja como una función para no ocultar su coste.
def benchmark_size(size: int):
    data = rng.random((size, d), dtype="float32")
    query = rng.random((nq, d), dtype="float32")
    exact = faiss.IndexFlatL2(d)
    exact.add(data)
    t0 = time.time(); exact.search(query, k); exact_ms = (time.time() - t0) / nq * 1000
    hnsw = faiss.IndexHNSWFlat(d, 16)
    hnsw.hnsw.efConstruction = 100
    t0 = time.time(); hnsw.add(data); build_s = time.time() - t0
    hnsw.hnsw.efSearch = 64
    t0 = time.time(); hnsw.search(query, k); hnsw_ms = (time.time() - t0) / nq * 1000
    return {"n": size, "exact_ms": exact_ms, "hnsw_ms": hnsw_ms, "build_s": build_s}

print("Para C1, ejecuta benchmark_size(100_000) y benchmark_size(500_000) según la memoria disponible.")
print("Conclusión: HNSW suele superar la fuerza bruta en latencia cuando el índice crece, a cambio de una aproximación y coste de construcción.")


## C4 · Evaluación automática de la recuperación

Necesitas las variables del notebook 03 (`retriever`, `all_docs`, …) o reconstruirlas aquí.

In [ ]:
import csv

def cargar_preguntas(ruta="eval/preguntas.csv"):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(retriever, filas):
    if not filas:
        raise ValueError("Se necesita al menos una pregunta de evaluación")
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        hits += any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata.get("source") == r["fuente_esperada"]
            for d in docs
        )
    return hits / len(filas)

print("Carga las 10 filas de eval/preguntas.csv y evalúa cada configuración de chunking con hit_at_k.")
print("Para comparar k=3 y k=5, crea el retriever correspondiente y registra ambos resultados.")


## C6 · Persistencia del índice en disco

Punto de partida: cambia el cliente en memoria por uno con ruta.

In [ ]:
from qdrant_client import QdrantClient
from pathlib import Path

qdrant_client = QdrantClient(path="./qdrant_data")
existe = qdrant_client.collection_exists("academic_papers_nvidia")
print("¿Ya existe la colección?", existe)

if existe:
    print("Índice persistido reutilizado; no se reindexan documentos.")
else:
    if "all_docs" not in globals() or "embeddings" not in globals():
        raise RuntimeError("Carga all_docs y embeddings antes de indexar el índice persistente")
    collection = "academic_papers_nvidia"
    dimension = len(embeddings.embed_query("dimension probe"))
    from qdrant_client.models import Distance, VectorParams
    qdrant_client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
    )
    from langchain_qdrant import QdrantVectorStore
    persistent_store = QdrantVectorStore(
        client=qdrant_client, collection_name=collection, embedding=embeddings
    )
    persistent_store.add_documents(all_docs)
    print(f"✅ {len(all_docs)} chunks indexados en disco")

print("Tras reiniciar el runtime, vuelve a ejecutar esta celda: collection_exists debe seguir siendo True.")
